Installing Libraries:

In [ ]:
!pip install opencv-python
!pip install matplotlib
!pip install numpy
!pip install scikit-learn

Libraries:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

Finding Height and Width of an Image:

In [ ]:
import cv2
import os

# Example sample image path from session 1
sample_image_path = '/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs/001/L_Fore/01.bmp'

# Load the image in grayscale
img = cv2.imread(sample_image_path, cv2.IMREAD_GRAYSCALE)

# Check if image was loaded successfully
if img is None:
    print("Image could not be loaded. Check the path.")
else:
    # Print its shape
    print("Image shape:", img.shape)

    # Print height and width
    height, width = img.shape
    print("Height:", height)
    print("Width:", width)

Preprocessing for Training:

In [ ]:
import os
import cv2
import numpy as np
from skimage import exposure
from tqdm import tqdm

# === CONFIGURATION ===
BASE_PATH = r"/content/drive/MyDrive/Datasets/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/MMCBNU_6000/ROIs"
IMAGE_SIZE = (128, 60)  # Per finger
FINGER_LIST = ['L_Fore', 'L_Middle', 'L_Ring', 'R_Fore', 'R_Middle', 'R_Ring']
NUM_IMAGES = 9  # Use first 9 images per finger for Protocol 3
FUSION_METHOD = 'vstack'  # Can also be 'hstack' or '2x3'

fused_images = []
fused_labels = []

# === STEP 1: LOAD & FUSE FINGER IMAGES ===
def load_and_fuse_finger_images(subject_path, subject_id):
    subject_samples = []
    labels = []

    for img_idx in range(1, NUM_IMAGES + 1):
        finger_images = []
        complete = True
        print(f"\n➡️ Subject {subject_id} - Image {img_idx:02d}")

        for finger in FINGER_LIST:
            img_path = os.path.join(subject_path, finger, f"{img_idx:02d}.bmp")
            print(f"  📥 Loading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"  ❌ Missing image: {img_path}")
                complete = False
                break

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
            finger_images.append(img_norm)

        if complete and len(finger_images) == 6:
            print(f"  🔄 Fusing 6 fingers using {FUSION_METHOD}")
            if FUSION_METHOD == 'vstack':
                fused_img = np.vstack(finger_images)
            elif FUSION_METHOD == 'hstack':
                fused_img = np.hstack(finger_images)
            elif FUSION_METHOD == '2x3':
                top = np.hstack(finger_images[:3])
                bottom = np.hstack(finger_images[3:])
                fused_img = np.vstack([top, bottom])
            else:
                raise ValueError("Unsupported fusion method.")

            subject_samples.append(fused_img)
            label = f"{subject_id}_img{img_idx:02d}"
            labels.append(label)
            print(f"  ✅ Fused image shape: {fused_img.shape}")
        else:
            print(f"  ⚠️ Skipped image {img_idx:02d}")

    return subject_samples, labels

# === STEP 2: LOAD ALL SUBJECTS ===
subject_dirs = sorted(os.listdir(BASE_PATH))
for subj in tqdm(subject_dirs, desc="Loading and Fusing Images"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    fused, labels = load_and_fuse_finger_images(subject_path, subj)
    fused_images.extend(fused)
    fused_labels.extend(labels)

print(f"\n✅ Total Fused Samples: {len(fused_images)}")
print(f"✅ Example Fused Image Shape: {fused_images[0].shape}")
fused_labels = np.array(fused_labels)

# === STEP 3: COMPUTE (2D)²PCA PROJECTION MATRICES (47×47) ===
def compute_2d2pca_projection(images, num_row_components, num_col_components):
    print("\n⚙️ Computing (2D)²PCA projection matrices...")
    n = len(images)
    h, w = images[0].shape
    mean_img = sum(images) / n

    G_row = np.zeros((h, h))
    G_col = np.zeros((w, w))

    for i, img in enumerate(images):
        A = img - mean_img
        G_row += A @ A.T
        G_col += A.T @ A
        if i < 3:
            print(f"  ➕ Sample {i+1} contribution added")

    G_row /= n
    G_col /= n

    eig_vals_r, eig_vecs_r = np.linalg.eigh(G_row)
    eig_vals_c, eig_vecs_c = np.linalg.eigh(G_col)

    idx_r = np.argsort(-eig_vals_r)
    idx_c = np.argsort(-eig_vals_c)

    U = eig_vecs_r[:, idx_r[:num_row_components]]  # Row projection
    V = eig_vecs_c[:, idx_c[:num_col_components]]  # Column projection

    print(f"✅ U (row) shape: {U.shape}, V (col) shape: {V.shape}")
    return U, V

# Fixed to 47×47 components
num_row_components = 47
num_col_components = 47

U, V = compute_2d2pca_projection(fused_images, num_row_components, num_col_components)

# === STEP 4: PROJECT FUSED IMAGES ===
projected_features = []
for i, img in enumerate(fused_images):
    feat = U.T @ img @ V
    projected_features.append(feat)
    if i < 3:
        print(f"🧮 Projected shape of sample {i+1}: {feat.shape}")

# === STEP 5: FLATTEN FOR CLASSIFIER (e.g., kNN, SVM) ===
flat_features = np.array([f.flatten() for f in projected_features])
print(f"\n✅ Final flattened feature matrix shape: {flat_features.shape}")
print(f"🧾 Number of training labels: {len(fused_labels)}")


Testing:

In [ ]:
# === TEST CONFIGURATION ===
TEST_INDICES = [10]  # Protocol 3 test images

test_images = []
test_labels = []
test_subject_dirs = sorted(os.listdir(BASE_PATH))

# === STEP 6: LOAD & FUSE TEST IMAGES ===
def load_and_fuse_test_images(subject_path, subject_id):
    subject_samples = []
    labels = []

    for img_idx in TEST_INDICES:
        finger_images = []
        complete = True
        print(f"\n📁 Test Subject {subject_id} - Image {img_idx:02d}")

        for finger in FINGER_LIST:
            img_path = os.path.join(subject_path, finger, f"{img_idx:02d}.bmp")
            print(f"  📥 Loading: {img_path}")
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
            if img is None:
                print(f"  ❌ Missing: {img_path}")
                complete = False
                break

            img = cv2.resize(img, IMAGE_SIZE)
            img_eq = exposure.equalize_hist(img).astype(np.float64)
            img_norm = (img_eq - np.mean(img_eq)) / (np.std(img_eq) + 1e-8)
            finger_images.append(img_norm)

        if complete and len(finger_images) == 6:
            if FUSION_METHOD == 'vstack':
                fused_img = np.vstack(finger_images)
            elif FUSION_METHOD == 'hstack':
                fused_img = np.hstack(finger_images)
            elif FUSION_METHOD == '2x3':
                top = np.hstack(finger_images[:3])
                bottom = np.hstack(finger_images[3:])
                fused_img = np.vstack([top, bottom])
            else:
                raise ValueError("Unsupported fusion method.")

            subject_samples.append(fused_img)
            labels.append(f"{subject_id}_img{img_idx:02d}")
            print(f"  ✅ Test fused image shape: {fused_img.shape}")
        else:
            print(f"  ⚠️ Incomplete fusion for image {img_idx:02d}")

    return subject_samples, labels

# === STEP 7: LOAD ALL TEST SUBJECTS ===
for subj in tqdm(test_subject_dirs, desc="Loading Test Set"):
    subject_path = os.path.join(BASE_PATH, subj)
    if not os.path.isdir(subject_path):
        continue

    fused_test_imgs, fused_test_lbls = load_and_fuse_test_images(subject_path, subj)
    test_images.extend(fused_test_imgs)
    test_labels.extend(fused_test_lbls)

test_labels = np.array(test_labels)

# === STEP 8: PROJECT TEST IMAGES USING (2D)²PCA ===
proj_test_features = []
for i, img in enumerate(test_images):
    feat = U.T @ img @ V
    proj_test_features.append(feat)
    if i < 3:
        print(f"🧪 Projected test sample {i+1} shape: {feat.shape}")

# === STEP 9: FLATTEN FOR CLASSIFIER ===
flat_test_features = np.array([f.flatten() for f in proj_test_features])
print(f"\n✅ Final test feature matrix shape: {flat_test_features.shape}")
print(f"🧾 Number of test labels: {len(test_labels)}")


Benchmarking:

In [ ]:
correct_matches = 0
total_tests = len(flat_test_features)

print("\n🔍 Starting classification using Manhattan distance...")

for i in range(total_tests):
    test_vec = flat_test_features[i]
    true_label = test_labels[i]  # e.g., "005_img08"

    # 📏 Compute Manhattan distance to all training vectors
    distances = np.sum(np.abs(flat_features - test_vec), axis=1)

    # 🏆 Find the closest match
    min_index = np.argmin(distances)
    predicted_label = fused_labels[min_index]  # e.g., "005_img03"

    print(f"\n🔹 Test sample {i+1}:")
    print(f"   🎯 Predicted → {predicted_label}")
    print(f"   ✅ Actual    → {true_label}")

    # Extract subject IDs only (everything before '_')
    pred_id = predicted_label.split('_')[0]
    true_id = true_label.split('_')[0]

    if pred_id == true_id:
        correct_matches += 1
        print("   🟢 Match (Subject ID correct)")
    else:
        print("   🔴 Mismatch")

# 📈 Compute final accuracy
accuracy = (correct_matches / total_tests) * 100
print(f"\n🏁 Final Recognition Accuracy: {accuracy:.2f}% ({correct_matches}/{total_tests})")
